This portion contains only the ViT model of the overall architecture. Used only for testing and have no significant changes from the final overall section

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import pandas as pd
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score


In [ ]:
#=== Loading train, test and val dataset ===

train_img_path_lst = []
train_labels_list = []

val_img_path_lst = []
val_labels_list = []

test_img_paths = []
test_labels_list = []


train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/skin disease encoded - Train.csv")

val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/skin disease encoded - Validation.csv")

test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/skin disease encoded - Test.csv")



for idx, row in train_df.iterrows():
    train_img_path_lst.append("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/" + row['image'])
    train_labels_list.append([row['name'],
                            row['cancerous'],
                            row['severity'],
                            row['cause'],
                            row['contagious'],
                            row['D0: Biopsy'],
                            row['D1: Dermatoscopy'],
                            row['D2: Imaging'],
                            row['D3: full skin examination'],
                            row['D4:  Blood test '],
                            row['D5: direct fluorescent antibody (DFA)'],
                            row['D6: clinical evaluation'],
                            row['D7: serological tests'],
                            row['D8: PCR test'],
                            row['D9: antigen detection test'],
                            row['D10: KOH Test'],
                            row['D11: Culture Test'],
                            row['P0: Vaccine'],
                            row['P1: Avoid UV'],
                            row['P2: Maintain hygiene'],
                            row['P3: Anitfungal Powder']])


for idx, row in val_df.iterrows():
    val_img_path_lst.append("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/" + row['image'])
    val_labels_list.append([row['name'],
                            row['cancerous'],
                            row['severity'],
                            row['cause'],
                            row['contagious'],
                            row['D0: Biopsy'],
                            row['D1: Dermatoscopy'],
                            row['D2: Imaging'],
                            row['D3: full skin examination'],
                            row['D4:  Blood test '],
                            row['D5: direct fluorescent antibody (DFA)'],
                            row['D6: clinical evaluation'],
                            row['D7: serological tests'],
                            row['D8: PCR test'],
                            row['D9: antigen detection test'],
                            row['D10: KOH Test'],
                            row['D11: Culture Test'],
                            row['P0: Vaccine'],
                            row['P1: Avoid UV'],
                            row['P2: Maintain hygiene'],
                            row['P3: Anitfungal Powder'],])



for idx, row in test_df.iterrows():
    test_img_paths.append("/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/" + row['image'])
    test_labels_list.append([row['name'],
                        row['cancerous'],
                        row['severity'],
                        row['cause'],
                        row['contagious'],
                        row['D0: Biopsy'],
                        row['D1: Dermatoscopy'],
                        row['D2: Imaging'],
                        row['D3: full skin examination'],
                        row['D4:  Blood test '],
                        row['D5: direct fluorescent antibody (DFA)'],
                        row['D6: clinical evaluation'],
                        row['D7: serological tests'],
                        row['D8: PCR test'],
                        row['D9: antigen detection test'],
                        row['D10: KOH Test'],
                        row['D11: Culture Test'],
                        row['P0: Vaccine'],
                        row['P1: Avoid UV'],
                        row['P2: Maintain hygiene'],
                        row['P3: Anitfungal Powder'],])




print("-------------train----------------")
for itm in train_img_path_lst:
  print(itm)

for lable in train_labels_list:
  print(lable)

print("-------------val--------------------")
for itm in val_img_path_lst:
  print(itm)

for lable in val_labels_list:
  print(lable)

print("-------------test--------------------")
for itm in test_img_paths:
  print(itm)

for lable in test_labels_list:
  print(lable)

-------------train----------------
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7623407.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7612427.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7612589.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7612992.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7613461.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7614036.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7614221.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7615896.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7623089.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7609643.jpg
/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combi

In [ ]:

# === Define your Multi-Head Model ===
class MultiHeadViT(nn.Module):
    def __init__(self, model_name, num_labels_list):
        super(MultiHeadViT, self).__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.heads = nn.ModuleList([
            nn.Linear(hidden_size, num_labels) for num_labels in num_labels_list
        ])

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        logits = [head(cls_token) for head in self.heads]
        return logits

# === Dummy Dataset ===
class DummyDataset(Dataset):
    def __init__(self, image_paths, labels_list, processor):
        self.image_paths = image_paths
        self.labels_list = labels_list
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")

        labels = [torch.tensor(x) for x in self.labels_list[idx]]  # FIXED

        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels': labels
        }


# === Instantiate everything ===
vis_model_name = "google/vit-base-patch16-224"
num_labels_list = [11, 2, 3, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

vis_model = MultiHeadViT(vis_model_name, num_labels_list)

processor = AutoImageProcessor.from_pretrained(vis_model_name, use_fast=True)


Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


#***Initialization (Default) Values :***

hidden_size
(The dimensionality of the hidden embeddings.) :
768

num_hidden_layers
(Number of transformer encoder layers.) :
12

num_attention_heads
(Number of attention heads in each layer.) :
12

intermediate_size
(Size of the feedforward layer in the transformer block) :
3072

patch_size
(Each image is split into patches of size n×n.) :
16

image_size
(Input image size.) :
224

num_channels
(Number of input channels.) :
3

attention_probs_dropout_prob
(Dropout rate applied to attention probabilities.) :
0.0

hidden_dropout_prob
(Dropout for hidden layers.) :
0.0

layer_norm_eps
(Epsilon for layer normalization.) :
1e-12

initializer_range
(Standard deviation of weight initializer.) :
0.02

hidden_act
(Activation function in the feed-forward layers.) :
gelu

pooler_act
(Activation function in pooled representation.) :
tanh

pooler_output_size
(Dimensionality of the output from the pooler.) :
16

qkv_bias
(Whether query/key/value projections use bias.) :
true




In [ ]:
print(vis_model.backbone.config.patch_size,
      vis_model.backbone.config.image_size,
      vis_model.backbone.config.hidden_dropout_prob,
      vis_model.backbone.config.attention_probs_dropout_prob,
      vis_model.backbone.config.hidden_act,
      vis_model.backbone.config.layer_norm_eps
)

16 224 0.0 0.0 gelu 1e-12


# Start Tuning

In [ ]:
vis_model.backbone.config.patch_size = 32  #keep in the 2^n(power of 2) format. ex : 16, 32.
vis_model.backbone.config.image_size = 224
vis_model.backbone.config.hidden_dropout_prob = 0.0
vis_model.backbone.config.attention_probs_dropout_prob = 0.0
vis_model.backbone.config.hidden_act = "gelu"
vis_model.backbone.config.layer_norm_eps = 2e-6  #smaller the better

epoch_size = 20
Batch_Sz = 4   #<---Batch Size

# Loss and optimizer

criterions = [nn.CrossEntropyLoss() for _ in num_labels_list]  # One loss per head

#optimizer = optim.Adam(vis_model.parameters(), lr=2e-5)                         #
optimizer = optim.AdamW(vis_model.parameters(), lr=2e-6, weight_decay=0.01)     #--->pick only one optimizer and change the learning rate
#optimizer = optim.Adagrad(vis_model.parameters(), lr=0.01)                    #--->and weight decay(if applicable) there directly
#optimizer = optim.RMSprop(vis_model.parameters(), lr=0.01)                    # p.s. if reducing lr, increase epoch


'''
For lr and layer_norm_eps:
popular values are 2e-5, 1e-12 but try more
'''

'\nFor lr and layer_norm_eps:\npopular values are 2e-5, 1e-12 but try more\n'

In [ ]:
print(vis_model.backbone.config.patch_size,
      vis_model.backbone.config.image_size,
      vis_model.backbone.config.hidden_dropout_prob,
      vis_model.backbone.config.attention_probs_dropout_prob,
      vis_model.backbone.config.hidden_act,
      vis_model.backbone.config.layer_norm_eps
)

NameError: name 'vis_model' is not defined

In [ ]:

train_dataset = DummyDataset(train_img_path_lst, train_labels_list, processor)
train_dataloader = DataLoader(train_dataset, batch_size=Batch_Sz, shuffle=True)

val_dataset = DummyDataset(val_img_path_lst, val_labels_list, processor)
val_dataloader = DataLoader(val_dataset, batch_size=Batch_Sz, shuffle=False)

test_dataset = DummyDataset(test_img_paths, test_labels_list, processor)
test_dataloader = DataLoader(test_dataset, batch_size=Batch_Sz, shuffle=False)


NameError: name 'DummyDataset' is not defined

In [ ]:
# === Training loop (Loading the model to the cpu/gpu to run)===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vis_model.to(device)

vis_model.train()

MultiHeadViT(
  (backbone): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act

In [ ]:
for epoch in range(epoch_size):
    vis_model.train()
    total_loss = 0
    correct_preds = [0] * len(num_labels_list)
    total_preds = [0] * len(num_labels_list)

    for batch in train_dataloader:
        pixel_values = batch['pixel_values'].to(device)
        labels = [label.to(device) for label in batch['labels']]

        optimizer.zero_grad()
        logits_list = vis_model(pixel_values)

        losses = []
        for i, (logits, label, criterion) in enumerate(zip(logits_list, labels, criterions)):
            loss = criterion(logits, label)
            losses.append(loss)

            preds = torch.argmax(logits, dim=1)
            correct_preds[i] += (preds == label).sum().item()
            total_preds[i] += label.size(0)

        loss = sum(losses)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Training Loss: {total_loss:.4f}")
    for i in range(len(num_labels_list)):
        acc = correct_preds[i] / total_preds[i] if total_preds[i] > 0 else 0
        print(f"  Train Task {i+1} Accuracy: {acc * 100:.2f}%")

    # === Validation after each epoch ===
    vis_model.eval()
    val_loss = 0
    val_correct_preds = [0] * len(num_labels_list)
    val_total_preds = [0] * len(num_labels_list)

    all_preds = [[] for _ in range(len(num_labels_list))]
    all_labels = [[] for _ in range(len(num_labels_list))]

    with torch.no_grad():
        for batch in val_dataloader:
            pixel_values = batch['pixel_values'].to(device)
            labels = [label.to(device) for label in batch['labels']]
            logits_list = vis_model(pixel_values)

            losses = []
            for i, (logits, label, criterion) in enumerate(zip(logits_list, labels, criterions)):
                loss = criterion(logits, label)
                losses.append(loss)

                preds = torch.argmax(logits, dim=1)

                val_correct_preds[i] += (preds == label).sum().item()
                val_total_preds[i] += label.size(0)

                all_preds[i].extend(preds.cpu().tolist())
                all_labels[i].extend(label.cpu().tolist())

            val_loss += sum(losses).item()

    print(f"          Validation Loss: {val_loss:.4f}")
    for i in range(len(num_labels_list)):
        acc = val_correct_preds[i] / val_total_preds[i] if val_total_preds[i] > 0 else 0
        precision = precision_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        recall = recall_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        f1 = f1_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        print(f"            Val Task {i+1} Accuracy: {acc * 100:.2f}%, Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")


Epoch 1 - Training Loss: 2874.1413
  Train Task 1 Accuracy: 19.81%
  Train Task 2 Accuracy: 74.19%
  Train Task 3 Accuracy: 51.26%
  Train Task 4 Accuracy: 19.33%
  Train Task 5 Accuracy: 83.79%
  Train Task 6 Accuracy: 72.51%
  Train Task 7 Accuracy: 61.10%
  Train Task 8 Accuracy: 67.83%
  Train Task 9 Accuracy: 75.63%
  Train Task 10 Accuracy: 92.32%
  Train Task 11 Accuracy: 87.15%
  Train Task 12 Accuracy: 74.43%
  Train Task 13 Accuracy: 77.31%
  Train Task 14 Accuracy: 89.68%
  Train Task 15 Accuracy: 84.87%
  Train Task 16 Accuracy: 92.92%
  Train Task 17 Accuracy: 84.27%
  Train Task 18 Accuracy: 91.12%
  Train Task 19 Accuracy: 78.03%
  Train Task 20 Accuracy: 84.15%
  Train Task 21 Accuracy: 81.15%
          Validation Loss: 293.8730
  Val Task 1 Accuracy: 25.49%
  Val Task 2 Accuracy: 76.47%
  Val Task 3 Accuracy: 58.82%
  Val Task 4 Accuracy: 44.12%
  Val Task 5 Accuracy: 92.16%
  Val Task 6 Accuracy: 67.65%
  Val Task 7 Accuracy: 71.57%
  Val Task 8 Accuracy: 91.18%
  Val

In [ ]:
def evaluate_model(model, dataloader, num_labels_list, device):
    model.eval()
    correct_preds = [0] * len(num_labels_list)
    total_preds = [0] * len(num_labels_list)

    all_preds = [[] for _ in range(len(num_labels_list))]
    all_labels = [[] for _ in range(len(num_labels_list))]

    with torch.no_grad():
        for batch in dataloader:
            pixel_values = batch['pixel_values'].to(device)
            labels = [label.to(device) for label in batch['labels']]

            logits_list = model(pixel_values)

            for i, (logits, label) in enumerate(zip(logits_list, labels)):
                preds = torch.argmax(logits, dim=1)

                correct_preds[i] += (preds == label).sum().item()
                total_preds[i] += label.size(0)

                all_preds[i].extend(preds.cpu().tolist())
                all_labels[i].extend(label.cpu().tolist())

    accuracies = [correct / total * 100 if total > 0 else 0.0
                  for correct, total in zip(correct_preds, total_preds)]

    precisions = []
    recalls = []
    f1s = []

    for i in range(len(num_labels_list)):
        precision = precision_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        recall = recall_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        f1 = f1_score(all_labels[i], all_preds[i], average='macro', zero_division=0)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)


    return accuracies, precisions, recalls, f1s


test_acc, test_prec, test_rec, test_f1 = evaluate_model(vis_model, test_dataloader, num_labels_list, device)

print("\n=== Test Set Metrics ===")
for i in range(len(num_labels_list)):
    print(f"Task {i+1} — Accuracy: {test_acc[i]:.2f}%, Precision: {test_prec[i]:.2f}, Recall: {test_rec[i]:.2f}, F1: {test_f1[i]:.2f}")



=== Test Set Accuracy ===
Task 1 Accuracy: 90.29%
Task 2 Accuracy: 98.06%
Task 3 Accuracy: 89.32%
Task 4 Accuracy: 98.06%
Task 5 Accuracy: 100.00%
Task 6 Accuracy: 77.67%
Task 7 Accuracy: 92.23%
Task 8 Accuracy: 86.41%
Task 9 Accuracy: 100.00%
Task 10 Accuracy: 96.12%
Task 11 Accuracy: 100.00%
Task 12 Accuracy: 83.50%
Task 13 Accuracy: 100.00%
Task 14 Accuracy: 95.15%
Task 15 Accuracy: 100.00%
Task 16 Accuracy: 93.20%
Task 17 Accuracy: 97.09%
Task 18 Accuracy: 100.00%
Task 19 Accuracy: 100.00%
Task 20 Accuracy: 97.09%
Task 21 Accuracy: 99.03%


In [ ]:
from PIL import Image
import torch

# === Set model to evaluation mode ===
vis_model.eval()

def vis_preds(img_path):

  # === Load and preprocess your image ===
  #img_path = "/content/drive/MyDrive/Skin Disease Dataset/dummy dataset/lab_pred.jpg"  # Your new image
  image = Image.open(img_path).convert("RGB")
  inputs = processor(images=image, return_tensors="pt")

  # === Move input to device (GPU/CPU) ===
  pixel_values = inputs['pixel_values'].to(device)

  # === Get model output ===
  with torch.no_grad():  # No gradient computation needed during inference
      logits_list = vis_model(pixel_values)

  # === Interpret outputs ===
  preds = []
  for logits in logits_list:
      pred_class = torch.argmax(logits, dim=-1)  # Take max logit for each head
      preds.append(pred_class.item())  # Convert to Python int

  return preds

img_path = '/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_7623089.jpg' # 0	1	2	0	0	1	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0
#img_path = '/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/_3_3279.jpg' # 3	0	1	3	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0	1	0
#img_path = '/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/26_BA-impetigo (73).jpg'  # 2	0	1	4	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0	1	0

print("Predicted classes:", vis_preds(img_path))

Predicted classes: [0, 1, 2, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]


In [ ]:
processor.save_pretrained("./vit_processor")

torch.save(vis_model.state_dict(), "multihead_vit_model_weights.pth")


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModel
from PIL import Image
import pandas as pd
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import torch.serialization

# === Define your Multi-Head Model ===
class MultiHeadViT(nn.Module):
    def __init__(self, model_name, num_labels_list):
        super(MultiHeadViT, self).__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.heads = nn.ModuleList([
            nn.Linear(hidden_size, num_labels) for num_labels in num_labels_list
        ])

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        cls_token = outputs.last_hidden_state[:, 0, :]
        logits = [head(cls_token) for head in self.heads]
        return logits

# === Dummy Dataset ===
class DummyDataset(Dataset):
    def __init__(self, image_paths, labels_list, processor):
        self.image_paths = image_paths
        self.labels_list = labels_list
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        inputs = self.processor(images=image, return_tensors="pt")

        labels = [torch.tensor(x) for x in self.labels_list[idx]]  # FIXED

        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels': labels
        }


# === Instantiate everything ===
vis_model_name = "google/vit-base-patch16-224"
num_labels_list = [11, 2, 3, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

#vis_model = MultiHeadViT(vis_model_name, num_labels_list)

processor = AutoImageProcessor.from_pretrained(vis_model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Re-initialize the model
# torch.serialization.add_safe_globals({'MultiHeadViT': MultiHeadViT})
#torch.serialization.add_safe_globals([MultiHeadViT])

# loaded_model = MultiHeadViT(model_name=vis_model_name, num_labels_list=num_labels_list)
# loaded_model.load_state_dict(torch.load("multihead_vit_model.pth"))
# loaded_model.load_state_dict(torch.load("multihead_vit_model_full.pt"))
#loaded_model = torch.load("multihead_vit_model_full.pt", map_location=device)
# loaded_model.to(device)
#loaded_model.eval()

#################
vis_model = MultiHeadViT(vis_model_name, num_labels_list)
vis_model.load_state_dict(torch.load("/content/t1_multihead_vit_model_weights (1).pth", map_location=device))
vis_model.to(device)
vis_model.eval()

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MultiHeadViT(
  (backbone): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_act

In [ ]:
print(vis_model.backbone.config.patch_size,
      vis_model.backbone.config.image_size,
      vis_model.backbone.config.hidden_dropout_prob,
      vis_model.backbone.config.attention_probs_dropout_prob,
      vis_model.backbone.config.hidden_act,
      vis_model.backbone.config.layer_norm_eps
)

#32 224 0.0 0.0 gelu 2e-06

16 224 0.0 0.0 gelu 1e-12


In [ ]:
vis_model.backbone.config.patch_size = 32  #keep in the 2^n(power of 2) format. ex : 16, 32.
vis_model.backbone.config.image_size = 224
vis_model.backbone.config.hidden_dropout_prob = 0.0
vis_model.backbone.config.attention_probs_dropout_prob = 0.0
vis_model.backbone.config.hidden_act = "gelu"
vis_model.backbone.config.layer_norm_eps = 2e-6  #smaller the better

epoch_size = 20
Batch_Sz = 4   #<---Batch Size

# Loss and optimizer

criterions = [nn.CrossEntropyLoss() for _ in num_labels_list]  # One loss per head

#optimizer = optim.Adam(vis_model.parameters(), lr=2e-5)                         #
optimizer = optim.AdamW(vis_model.parameters(), lr=2e-6, weight_decay=0.01)     #--->pick only one optimizer and change the learning rate
#optimizer = optim.Adagrad(vis_model.parameters(), lr=0.01)                    #--->and weight decay(if applicable) there directly
#optimizer = optim.RMSprop(vis_model.parameters(), lr=0.01)                    # p.s. if reducing lr, increase epoch


In [ ]:
print(vis_model.backbone.config.patch_size,
      vis_model.backbone.config.image_size,
      vis_model.backbone.config.hidden_dropout_prob,
      vis_model.backbone.config.attention_probs_dropout_prob,
      vis_model.backbone.config.hidden_act,
      vis_model.backbone.config.layer_norm_eps
)

32 224 0.0 0.0 gelu 2e-06


In [ ]:
train_dataset = DummyDataset(train_img_path_lst, train_labels_list, processor)
train_dataloader = DataLoader(train_dataset, batch_size=Batch_Sz, shuffle=True)

val_dataset = DummyDataset(val_img_path_lst, val_labels_list, processor)
val_dataloader = DataLoader(val_dataset, batch_size=Batch_Sz, shuffle=False)

test_dataset = DummyDataset(test_img_paths, test_labels_list, processor)
test_dataloader = DataLoader(test_dataset, batch_size=Batch_Sz, shuffle=False)


In [ ]:
img_path = '/content/drive/MyDrive/Colab Notebooks/Skin disease Dataset/combined-img/ISIC_6753148.jpg'  # 0	1	2	0	0	1	1	0	0	0	0	0	0	0	0	0	0	0	1	0	0

print("Predicted classes:", vis_preds(img_path))

NameError: name 'vis_preds' is not defined

In [ ]:
def evaluate_model(model, dataloader, num_labels_list, device):
    model.eval()
    correct_preds = [0] * len(num_labels_list)
    total_preds = [0] * len(num_labels_list)

    all_preds = [[] for _ in range(len(num_labels_list))]
    all_labels = [[] for _ in range(len(num_labels_list))]

    with torch.no_grad():
        for batch in dataloader:
            pixel_values = batch['pixel_values'].to(device)
            labels = [label.to(device) for label in batch['labels']]

            logits_list = model(pixel_values)

            for i, (logits, label) in enumerate(zip(logits_list, labels)):
                preds = torch.argmax(logits, dim=1)

                correct_preds[i] += (preds == label).sum().item()
                total_preds[i] += label.size(0)

                all_preds[i].extend(preds.cpu().tolist())
                all_labels[i].extend(label.cpu().tolist())

    accuracies = [correct / total * 100 if total > 0 else 0.0
                  for correct, total in zip(correct_preds, total_preds)]

    precisions = []
    recalls = []
    f1s = []

    for i in range(len(num_labels_list)):
        precision = precision_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        recall = recall_score(all_labels[i], all_preds[i], average='macro', zero_division=0)
        f1 = f1_score(all_labels[i], all_preds[i], average='macro', zero_division=0)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)


    return accuracies, precisions, recalls, f1s


test_acc, test_prec, test_rec, test_f1 = evaluate_model(vis_model, test_dataloader, num_labels_list, device)

print("\n=== Test Set Metrics ===")
for i in range(len(num_labels_list)):
    print(f"Task {i+1} — Accuracy: {test_acc[i]:.2f}%, Precision: {test_prec[i]:.2f}, Recall: {test_rec[i]:.2f}, F1: {test_f1[i]:.2f}")



=== Test Set Metrics ===
Task 1 — Accuracy: 90.29%, Precision: 0.92, Recall: 0.92, F1: 0.92
Task 2 — Accuracy: 95.15%, Precision: 0.95, Recall: 0.95, F1: 0.95
Task 3 — Accuracy: 89.32%, Precision: 0.89, Recall: 0.87, F1: 0.88
Task 4 — Accuracy: 98.06%, Precision: 0.98, Recall: 0.98, F1: 0.98
Task 5 — Accuracy: 100.00%, Precision: 1.00, Recall: 1.00, F1: 1.00
Task 6 — Accuracy: 79.61%, Precision: 0.80, Recall: 0.78, F1: 0.78
Task 7 — Accuracy: 91.26%, Precision: 0.93, Recall: 0.86, F1: 0.89
Task 8 — Accuracy: 83.50%, Precision: 0.47, Recall: 0.44, F1: 0.46
Task 9 — Accuracy: 100.00%, Precision: 1.00, Recall: 1.00, F1: 1.00
Task 10 — Accuracy: 96.12%, Precision: 0.48, Recall: 0.50, F1: 0.49
Task 11 — Accuracy: 100.00%, Precision: 1.00, Recall: 1.00, F1: 1.00
Task 12 — Accuracy: 82.52%, Precision: 0.51, Recall: 0.51, F1: 0.50
Task 13 — Accuracy: 100.00%, Precision: 1.00, Recall: 1.00, F1: 1.00
Task 14 — Accuracy: 97.09%, Precision: 0.79, Recall: 0.98, F1: 0.86
Task 15 — Accuracy: 100.00%

In [ ]:
#